In [1]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")

CLIENT_ID = os.getenv("DB_CLIENT_ID")
API_KEY = os.getenv("DB_API_KEY")

print("Client ID:", CLIENT_ID[:8] + "..." if CLIENT_ID else "MISSING")
print("API Key:", "loaded" if API_KEY else "MISSING")

Client ID: bb989c0e...
API Key: loaded


In [2]:
import requests

BASE_URL = "https://apis.deutschebahn.com/db-api-marketplace/apis/timetables/v1"

headers = {
    "DB-Client-Id": CLIENT_ID,
    "DB-Api-Key": API_KEY,
    "Accept": "application/xml",
}

# Step 1: find Berlin Hbf's eva number
response = requests.get(
    f"{BASE_URL}/station/BLS",   # BLS = official DS100 code for Berlin Hbf
    headers=headers,
    timeout=30,
)

print("Status:", response.status_code)
print("Content-Type:", response.headers.get("content-type"))
print("First 500 chars:")
print(response.text[:500])

Status: 200
Content-Type: application/xml;charset=UTF-8
First 500 chars:
<stations>

<station p="11|12 D - G|12|13 A - D|13|14|13 D - G|13 A - C|13 C - D|11 D - G|14 A - D|14 A - C|14 C - D|14 E - F|11 C - D|13 E - F|11 E - F|11 A - D|12 A - D|14 D - G|12 C - D|12 E - F" meta="8070952|8089021|8098160" name="Berlin Hbf" eva="8011160" ds100="BLS" db="true" creationts="26-05-08 10:15:24.735"/>

</stations>



In [3]:
from datetime import datetime

EVA_BERLIN_HBF = "8011160"

now = datetime.now()
date_str = now.strftime("%y%m%d")   # e.g. "260515"
hour_str = now.strftime("%H")       # e.g. "14"

print(f"Fetching planned timetable for date={date_str} hour={hour_str}")

response = requests.get(
    f"{BASE_URL}/plan/{EVA_BERLIN_HBF}/{date_str}/{hour_str}",
    headers=headers,
    timeout=30,
)

print("Status:", response.status_code)
print("Length:", len(response.text), "chars")
print("First 1500 chars:")
print(response.text[:1500])

Fetching planned timetable for date=260515 hour=21
Status: 200
Length: 5373 chars
First 1500 chars:
<?xml version='1.0' encoding='UTF-8'?><timetable station='Berlin Hbf'><s id="-3963019265143400528-2605152044-9"><tl f="D" t="p" o="OWRE" c="OE" n="73785"/><ar pt="2605152135" pp="12" l="RE1" fb="RE1" ppth="Brandenburg Hbf|Werder(Havel)|Potsdam Park Sanssouci|Potsdam Charlottenhof|Potsdam Hbf|Berlin-Wannsee|Berlin-Charlottenburg|Berlin Zoologischer Garten"/><dp pt="2605152137" pp="12" l="RE1" fb="RE1" ppth="Berlin Friedrichstraße|Berlin Alexanderplatz|Berlin Ostbahnhof|Berlin Ostkreuz|Erkner|Fürstenwalde(Spree)|Frankfurt(Oder)"/></s><s id="5874878309093991832-2605151416-17"><tl f="F" t="p" o="80" c="ICE" n="278"/><ar pt="2605152131" pp="11" fb="ICE 278" ppth="Basel SBB|Basel Bad Bf|Freiburg(Breisgau) Hbf|Offenburg|Karlsruhe Hbf|Mannheim Hbf|Frankfurt(Main)Hbf|Hanau Hbf|Fulda|Kassel-Wilhelmshöhe|Göttingen|Hildesheim Hbf|Braunschweig Hbf|Wolfsburg Hbf|Berlin-Spandau|Berlin Zoologischer Gart

In [4]:
response = requests.get(
    f"{BASE_URL}/fchg/{EVA_BERLIN_HBF}",
    headers=headers,
    timeout=30,
)

print("Status:", response.status_code)
print("Length:", len(response.text), "chars")
print("First 1500 chars:")
print(response.text[:1500])

Status: 200
Length: 165882 chars
First 1500 chars:
<timetable station="Berlin Hbf" eva="8011160">

<s id="3272196155422799371-2605151101-17" eva="8011160">
    <m id="r2621054" t="h" from="2605151101" to="2605151942" cat="Information" ts="2605151120" ts-tts="26-05-15 11:20:19.320" pr="2"/>
    <m id="r2604793" t="h" from="2604140000" to="2605312359" cat="Information" ts="2604141735" ts-tts="26-05-14 23:16:04.763" pr="2"/>
    <m id="r2621115" t="h" from="2605151300" to="2605151600" cat="Störung" ts="2605151402" ts-tts="26-05-15 14:02:15.785" pr="1"/>
    <m id="r2621148" t="h" from="2605151400" to="2605151800" cat="Störung" ts="2605151452" ts-tts="26-05-15 17:16:04.090" pr="1"/>
    <m id="r2621149" t="h" from="2605151400" to="2605151600" cat="Störung" ts="2605151457" ts-tts="26-05-15 14:57:53.861" pr="1"/>
    <m id="r1588418959" t="c" ts="2605152005" ts-tts="26-05-15 20:05:02.026"/>
    <m id="r1588418960" t="c" ts="2605152005" ts-tts="26-05-15 20:05:02.026"/>
    <m id="r1588418961"

In [5]:
import pandas as pd
import xml.etree.ElementTree as ET
from datetime import datetime

def parse_db_time(s):
    """DB times are YYMMDDHHMM strings, e.g. '2605151644' = 2026-05-15 16:44"""
    if not s:
        return None
    return datetime.strptime(s, "%y%m%d%H%M")

def parse_plan_xml(xml_text):
    """Parse a /plan/... response into a list of dicts."""
    root = ET.fromstring(xml_text)
    rows = []
    for s in root.findall("s"):
        stop_id = s.get("id")
        tl = s.find("tl")
        ar = s.find("ar")
        dp = s.find("dp")

        row = {
            "stop_id": stop_id,
            "train_category": tl.get("c") if tl is not None else None,
            "train_number": tl.get("n") if tl is not None else None,
            "operator": tl.get("o") if tl is not None else None,
            "planned_arrival": parse_db_time(ar.get("pt")) if ar is not None else None,
            "planned_arrival_platform": ar.get("pp") if ar is not None else None,
            "arrival_line": ar.get("l") if ar is not None else None,
            "arrival_path": ar.get("ppth") if ar is not None else None,
            "planned_departure": parse_db_time(dp.get("pt")) if dp is not None else None,
            "planned_departure_platform": dp.get("pp") if dp is not None else None,
            "departure_line": dp.get("l") if dp is not None else None,
            "departure_path": dp.get("ppth") if dp is not None else None,
        }
        rows.append(row)
    return rows


# Re-fetch the plan and parse
date_str = datetime.now().strftime("%y%m%d")
hour_str = datetime.now().strftime("%H")

plan_response = requests.get(
    f"{BASE_URL}/plan/{EVA_BERLIN_HBF}/{date_str}/{hour_str}",
    headers=headers,
    timeout=30,
)

plan_rows = parse_plan_xml(plan_response.text)
plan_df = pd.DataFrame(plan_rows)

print(f"Got {len(plan_df)} scheduled stops at Berlin Hbf for hour {hour_str}")
plan_df.head(10)

Got 11 scheduled stops at Berlin Hbf for hour 21


,stop_id,train_category,train_number,operator,planned_arrival,planned_arrival_platform,arrival_line,arrival_path,planned_departure,planned_departure_platform,departure_line,departure_path
0,-3963019265143400528-2605152044-9,OE,73785,OWRE,2026-05-15 21:35:00,12,RE1,Brandenburg Hbf|Werder(Havel)|Potsdam Park San...,2026-05-15 21:37:00,12,RE1,Berlin Friedrichstraße|Berlin Alexanderplatz|B...
1,5874878309093991832-2605151416-17,ICE,278,80,2026-05-15 21:31:00,11,NaN,Basel SBB|Basel Bad Bf|Freiburg(Breisgau) Hbf|...,2026-05-15 21:34:00,11,NaN,Berlin Ostbahnhof
2,8679325635361758951-2605151626-15,ICE,643,80,2026-05-15 21:18:00,12,NaN,Köln Hbf|Düsseldorf Hbf|Düsseldorf Flughafen|D...,2026-05-15 21:22:00,12,NaN,Berlin Ostbahnhof
3,3941548061613216885-2605152112-1,D,300,743051,NaT,NaN,NaN,NaN,2026-05-15 21:12:00,14,NaN,Hamburg Hbf|Koebenhavn Syd|Malmö Central
4,3089420339552489300-2605152032-15,OE,73790,OWRE,2026-05-15 21:56:00,14,RE1,Frankfurt(Oder)|Frankfurt(Oder)-Rosengarten|Pi...,2026-05-15 21:58:00,14,RE1,Berlin Zoologischer Garten|Berlin-Wannsee|Pots...
5,5920790384366355547-2605151600-13,ICE,241,84,2026-05-15 21:51:00,11,NaN,Amsterdam Centraal|Hilversum|Amersfoort Centra...,2026-05-15 21:55:00,11,NaN,Berlin Ostbahnhof
6,6602161626755380743-2605151942-14,OE,73748,OWRE,2026-05-15 21:02:00,14,RE1,Frankfurt(Oder)|Frankfurt(Oder)-Rosengarten|Pi...,2026-05-15 21:04:00,14,RE1,Berlin Zoologischer Garten|Berlin-Charlottenbu...
7,2121877044531283563-2605152132-2,ICE,840,80,2026-05-15 21:42:00,14,NaN,Berlin Ostbahnhof,2026-05-15 21:44:00,14,NaN,Berlin Zoologischer Garten|Berlin-Spandau|Sten...
8,-7807139520157065993-2605152005-18,RE,3736,800165,2026-05-15 21:44:00,12,RE7,Dessau Hbf|Roßlau(Elbe)|Jeber-Bergfrieden|Wies...,2026-05-15 21:46:00,12,RE7,Berlin Friedrichstraße|Berlin Alexanderplatz|B...
9,7749992071955110630-2605152104-3,OE,73825,OWRE,2026-05-15 21:13:00,11,RE1,Berlin-Charlottenburg|Berlin Zoologischer Garten,2026-05-15 21:15:00,11,RE1,Berlin Friedrichstraße|Berlin Alexanderplatz|B...


In [6]:
fchg_response = requests.get(
    f"{BASE_URL}/fchg/{EVA_BERLIN_HBF}",
    headers=headers,
    timeout=30,
)

print("Status:", fchg_response.status_code)
print("Length:", len(fchg_response.text), "chars")
print("First 2000 chars:")
print(fchg_response.text[:2000])

Status: 200
Length: 165882 chars
First 2000 chars:
<timetable station="Berlin Hbf" eva="8011160">

<s id="3272196155422799371-2605151101-17" eva="8011160">
    <m id="r2621054" t="h" from="2605151101" to="2605151942" cat="Information" ts="2605151120" ts-tts="26-05-15 11:20:19.320" pr="2"/>
    <m id="r2604793" t="h" from="2604140000" to="2605312359" cat="Information" ts="2604141735" ts-tts="26-05-14 23:16:04.763" pr="2"/>
    <m id="r2621115" t="h" from="2605151300" to="2605151600" cat="Störung" ts="2605151402" ts-tts="26-05-15 14:02:15.785" pr="1"/>
    <m id="r2621148" t="h" from="2605151400" to="2605151800" cat="Störung" ts="2605151452" ts-tts="26-05-15 17:16:04.090" pr="1"/>
    <m id="r2621149" t="h" from="2605151400" to="2605151600" cat="Störung" ts="2605151457" ts-tts="26-05-15 14:57:53.861" pr="1"/>
    <m id="r1588418959" t="c" ts="2605152005" ts-tts="26-05-15 20:05:02.026"/>
    <m id="r1588418960" t="c" ts="2605152005" ts-tts="26-05-15 20:05:02.026"/>
    <m id="r1588418961"

In [7]:
def parse_fchg_xml(xml_text):
    """Parse a /fchg/... response into a list of dicts."""
    root = ET.fromstring(xml_text)
    rows = []
    for s in root.findall("s"):
        stop_id = s.get("id")
        ar = s.find("ar")
        dp = s.find("dp")

        # collect delay reason codes
        reasons = []
        for m in s.iter("m"):
            t = m.get("t")
            c = m.get("c")
            if t == "d" and c:   # delay code
                reasons.append(c)

        row = {
            "stop_id": stop_id,
            "changed_arrival": parse_db_time(ar.get("ct")) if ar is not None and ar.get("ct") else None,
            "changed_arrival_platform": ar.get("cp") if ar is not None else None,
            "changed_departure": parse_db_time(dp.get("ct")) if dp is not None and dp.get("ct") else None,
            "changed_departure_platform": dp.get("cp") if dp is not None else None,
            "delay_codes": ",".join(sorted(set(reasons))) if reasons else None,
        }
        rows.append(row)
    return rows


fchg_rows = parse_fchg_xml(fchg_response.text)
fchg_df = pd.DataFrame(fchg_rows)

print(f"Got {len(fchg_df)} changed stops")
fchg_df.head(10)

Got 212 changed stops


,stop_id,changed_arrival,changed_arrival_platform,changed_departure,changed_departure_platform,delay_codes
0,3272196155422799371-2605151101-17,2026-05-15 20:32:00,NaN,2026-05-15 20:35:00,NaN,"38,55"
1,840720102773351368-2605151536-10,2026-05-15 20:58:00,NaN,2026-05-15 21:02:00,NaN,"38,43"
2,8082573176163790166-2605151935-6,2026-05-15 20:24:00,NaN,2026-05-15 20:27:00,NaN,NaN
3,1923081405738379549-2605151905-19,2026-05-15 20:50:00,NaN,2026-05-15 20:52:00,NaN,64
4,5874878309093991832-2605151416-17,2026-05-15 21:39:00,NaN,2026-05-15 21:42:00,NaN,NaN
5,2869769973922245820-2605152020-2,2026-05-15 20:28:00,NaN,2026-05-15 20:33:00,NaN,NaN
6,265997371841578616-2605151944-9,2026-05-15 20:36:00,NaN,2026-05-15 20:38:00,NaN,NaN
7,-956811204558686568-2605151400-13,2026-05-15 20:27:00,12,2026-05-15 20:29:00,12,"38,55"
8,3089420339552489300-2605152032-15,2026-05-15 21:56:00,NaN,2026-05-15 21:58:00,NaN,NaN
9,-5637803255857346220-2605151528-11,2026-05-15 20:46:00,NaN,2026-05-15 20:50:00,NaN,"48,7"


In [8]:
# Join planned with changes by stop_id
merged = plan_df.merge(fchg_df, on="stop_id", how="left")

# Compute delays in minutes
merged["arrival_delay_min"] = (
    (merged["changed_arrival"] - merged["planned_arrival"]).dt.total_seconds() / 60
)
merged["departure_delay_min"] = (
    (merged["changed_departure"] - merged["planned_departure"]).dt.total_seconds() / 60
)

# Sort by departure delay, biggest first
delayed = (
    merged[merged["departure_delay_min"].notna() & (merged["departure_delay_min"] != 0)]
    .sort_values("departure_delay_min", ascending=False)
    [["train_category", "train_number", "departure_line",
      "planned_departure", "changed_departure", "departure_delay_min",
      "delay_codes"]]
)

print(f"{len(delayed)} delayed trains right now")
delayed.head(20)

1 delayed trains right now


,train_category,train_number,departure_line,planned_departure,changed_departure,departure_delay_min,delay_codes
1,ICE,278,NaN,2026-05-15 21:34:00,2026-05-15 21:42:00,8.0,NaN


In [9]:
from datetime import timedelta

def fetch_plan_window(eva, hours_ahead=6):
    """Fetch /plan for the current hour + the next N hours."""
    all_rows = []
    now = datetime.now()
    for offset in range(hours_ahead + 1):
        t = now + timedelta(hours=offset)
        date_str = t.strftime("%y%m%d")
        hour_str = t.strftime("%H")
        r = requests.get(
            f"{BASE_URL}/plan/{eva}/{date_str}/{hour_str}",
            headers=headers,
            timeout=30,
        )
        if r.status_code == 200:
            all_rows.extend(parse_plan_xml(r.text))
        else:
            print(f"Hour {hour_str}: status {r.status_code}")
    return all_rows


plan_rows = fetch_plan_window(EVA_BERLIN_HBF, hours_ahead=6)
plan_df = pd.DataFrame(plan_rows)
print(f"Got {len(plan_df)} planned stops over the next 7 hours")

Got 50 planned stops over the next 7 hours


In [10]:
merged = plan_df.merge(fchg_df, on="stop_id", how="left")
merged["departure_delay_min"] = (
    (merged["changed_departure"] - merged["planned_departure"]).dt.total_seconds() / 60
)

delayed = (
    merged[merged["departure_delay_min"].notna() & (merged["departure_delay_min"] != 0)]
    .sort_values("departure_delay_min", ascending=False)
    [["train_category", "train_number", "departure_line",
      "planned_departure", "changed_departure", "departure_delay_min",
      "delay_codes"]]
)

print(f"{len(delayed)} delayed trains")
delayed.head(20)

2 delayed trains


,train_category,train_number,departure_line,planned_departure,changed_departure,departure_delay_min,delay_codes
14,ICE,792,NaN,2026-05-15 22:42:00,2026-05-15 23:16:00,34.0,48
1,ICE,278,NaN,2026-05-15 21:34:00,2026-05-15 21:42:00,8.0,NaN


In [11]:
import os
from sqlalchemy import create_engine, text

PG_USER = os.getenv("DB_PG_USER")
PG_PASSWORD = os.getenv("DB_PG_PASSWORD")
PG_HOST = os.getenv("DB_PG_HOST")
PG_PORT = os.getenv("DB_PG_PORT")
PG_DB = os.getenv("DB_PG_DATABASE")

engine = create_engine(
    f"postgresql+psycopg://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}"
)

with engine.connect() as conn:
    result = conn.execute(text("SELECT version();"))
    print(result.scalar())

PostgreSQL 16.13 (Ubuntu 16.13-0ubuntu0.24.04.1) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit


In [12]:
from sqlalchemy import text

create_tables_sql = """
CREATE TABLE IF NOT EXISTS stations (
    eva                 TEXT PRIMARY KEY,
    name                TEXT NOT NULL,
    ds100               TEXT,
    created_at          TIMESTAMP DEFAULT NOW()
);

CREATE TABLE IF NOT EXISTS planned_stops (
    stop_id                     TEXT NOT NULL,
    station_eva                 TEXT NOT NULL,
    train_category              TEXT,
    train_number                TEXT,
    operator                    TEXT,
    planned_arrival             TIMESTAMP,
    planned_arrival_platform    TEXT,
    arrival_line                TEXT,
    arrival_path                TEXT,
    planned_departure           TIMESTAMP,
    planned_departure_platform  TEXT,
    departure_line              TEXT,
    departure_path              TEXT,
    fetched_at                  TIMESTAMP DEFAULT NOW(),
    PRIMARY KEY (stop_id, station_eva)
);

CREATE TABLE IF NOT EXISTS changed_stops (
    stop_id                     TEXT NOT NULL,
    station_eva                 TEXT NOT NULL,
    changed_arrival             TIMESTAMP,
    changed_arrival_platform    TEXT,
    changed_departure           TIMESTAMP,
    changed_departure_platform  TEXT,
    delay_codes                 TEXT,
    fetched_at                  TIMESTAMP NOT NULL DEFAULT NOW(),
    PRIMARY KEY (stop_id, station_eva, fetched_at)
);
"""

with engine.begin() as conn:
    conn.execute(text(create_tables_sql))

print("Tables created.")

Tables created.


In [13]:
import pandas as pd

stations_df = pd.DataFrame([{
    "eva": "8011160",
    "name": "Berlin Hbf",
    "ds100": "BLS",
}])

with engine.begin() as conn:
    # Clear and reinsert (idempotent for now; we'll handle conflicts properly later)
    conn.execute(text("DELETE FROM stations WHERE eva = :eva"), {"eva": "8011160"})

stations_df.to_sql("stations", engine, if_exists="append", index=False)
print("Berlin Hbf added.")

Berlin Hbf added.


In [14]:
# Prepare planned_stops with station_eva column
plan_to_write = plan_df.copy()
plan_to_write["station_eva"] = "8011160"

# Postgres can't take pandas NaN/NaT — convert to None
plan_to_write = plan_to_write.astype(object).where(plan_to_write.notna(), None)

with engine.begin() as conn:
    conn.execute(text("DELETE FROM planned_stops WHERE station_eva = :eva"),
                 {"eva": "8011160"})

plan_to_write.to_sql("planned_stops", engine, if_exists="append", index=False)
print(f"Wrote {len(plan_to_write)} rows to planned_stops")

# Same for changed_stops
fchg_to_write = fchg_df.copy()
fchg_to_write["station_eva"] = "8011160"
fchg_to_write = fchg_to_write.astype(object).where(fchg_to_write.notna(), None)

fchg_to_write.to_sql("changed_stops", engine, if_exists="append", index=False)
print(f"Wrote {len(fchg_to_write)} rows to changed_stops")


Wrote 50 rows to planned_stops
Wrote 212 rows to changed_stops


In [15]:
with engine.connect() as conn:
    stations_count = conn.execute(text("SELECT COUNT(*) FROM stations")).scalar()
    plan_count = conn.execute(text("SELECT COUNT(*) FROM planned_stops")).scalar()
    chg_count = conn.execute(text("SELECT COUNT(*) FROM changed_stops")).scalar()
    
print(f"stations: {stations_count} rows")
print(f"planned_stops: {plan_count} rows")
print(f"changed_stops: {chg_count} rows")

stations: 1 rows
planned_stops: 50 rows
changed_stops: 1307 rows


In [16]:
query = """
SELECT 
    p.train_category,
    p.train_number,
    p.departure_line,
    p.planned_departure,
    c.changed_departure,
    EXTRACT(EPOCH FROM (c.changed_departure - p.planned_departure)) / 60 AS delay_min,
    c.delay_codes
FROM planned_stops p
LEFT JOIN changed_stops c ON p.stop_id = c.stop_id AND p.station_eva = c.station_eva
WHERE c.changed_departure IS NOT NULL
  AND c.changed_departure != p.planned_departure
ORDER BY delay_min DESC NULLS LAST
LIMIT 20;
"""

result = pd.read_sql(query, engine)
result

,train_category,train_number,departure_line,planned_departure,changed_departure,delay_min,delay_codes
0,ICE,792,NaN,2026-05-15 22:42:00,2026-05-15 23:16:00,34.0,48
1,OE,73748,RE1,2026-05-15 21:04:00,2026-05-15 21:33:00,29.0,NaN
2,ICE,792,NaN,2026-05-15 22:42:00,2026-05-15 23:05:00,23.0,48
3,ICE,792,NaN,2026-05-15 22:42:00,2026-05-15 23:05:00,23.0,48
4,ICE,792,NaN,2026-05-15 22:42:00,2026-05-15 23:05:00,23.0,48
5,ICE,792,NaN,2026-05-15 22:42:00,2026-05-15 22:58:00,16.0,48
6,ICE,278,NaN,2026-05-15 21:34:00,2026-05-15 21:42:00,8.0,NaN
7,ICE,643,NaN,2026-05-15 21:22:00,2026-05-15 21:25:00,3.0,NaN
8,ICE,643,NaN,2026-05-15 21:22:00,2026-05-15 21:24:00,2.0,NaN
9,ICE,643,NaN,2026-05-15 21:22:00,2026-05-15 21:24:00,2.0,NaN


In [17]:
pd.read_sql("""
SELECT 
    COUNT(*) AS total_planned_stops,
    COUNT(c.changed_departure) AS stops_with_changes,
    AVG(EXTRACT(EPOCH FROM (c.changed_departure - p.planned_departure)) / 60) AS avg_delay_min,
    MAX(EXTRACT(EPOCH FROM (c.changed_departure - p.planned_departure)) / 60) AS max_delay_min
FROM planned_stops p
LEFT JOIN changed_stops c ON p.stop_id = c.stop_id;
""", engine)

,total_planned_stops,stops_with_changes,avg_delay_min,max_delay_min
0,199,112,1.473214,34.0


In [18]:
import requests
from db_delays.config import DB_API_BASE_URL, DB_API_HEADERS

for ds100 in ["BLS", "MH", "FF", "KK"]:
    r = requests.get(
        f"{DB_API_BASE_URL}/station/{ds100}",
        headers=DB_API_HEADERS,
        timeout=30,
    )
    print(ds100, "→ status:", r.status_code)
    print(r.text[:200])
    print("---")

BLS → status: 200
<stations>

<station p="11|12 D - G|12|13 A - D|13|14|13 D - G|13 A - C|13 C - D|11 D - G|14 A - D|14 A - C|14 C - D|14 E - F|11 C - D|13 E - F|11 E - F|11 A - D|12 A - D|14 D - G|12 C - D|12 E - F" m
---
MH → status: 200
<stations>

<station meta="270002|8098261|8098262|8098263" name="München Hbf" eva="8000261" ds100="MH" db="true" creationts="26-05-08 10:43:29.206"/>

</stations>

---
FF → status: 200
<stations>

<station p="22|23|24|1a|10|11|12|13|14|15|16|17|18|19|1|2|3|4|5|6|7|8|9|20|21" meta="8089211|8098105" name="Frankfurt(Main)Hbf" eva="8000105" ds100="FF" db="true" creationts="26-05-08 10:1
---
KK → status: 200
<stations>

<station meta="151200|161264|180220|391612|8089322" name="Köln Hbf" eva="8000207" ds100="KK" db="true" creationts="26-05-08 10:43:29.187"/>

</stations>

---
